## 导入库和全局设置

In [3]:
# 导入必要的库
import csv
import json
import os
import re


In [5]:
import pandas as pd

from typing import Any, Dict, List, Tuple
from datetime import datetime

try:
    from openpyxl import Workbook
    from openpyxl.styles import Font, PatternFill
    EXCEL_AVAILABLE = True
except ImportError:
    EXCEL_AVAILABLE = False
    print("警告: 未安装openpyxl库，将使用CSV格式输出")

ModuleNotFoundError: No module named 'pandas'

## 工具函数定义

In [71]:
def read_csv_with_encoding(file_path: str) -> Tuple[List[str], List[List[str]]]:
    """通用CSV文件读取函数，自动尝试多种编码"""
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"文件不存在: {file_path}")
    
    encodings = ["utf-8", "gbk", "gb2312", "latin-1", "cp1252", "utf-8-sig"]
    
    for encoding in encodings:
        try:
            headers = []
            rows = []
            with open(file_path, "r", encoding=encoding) as f:
                reader = csv.reader(f)
                headers = next(reader)
                for row in reader:
                    rows.append(row)
            
            print(f"文件读取成功: {file_path}, 使用编码: {encoding}, 共 {len(rows)} 行")
            return headers, rows
        except UnicodeDecodeError:
            continue
        except Exception as e:
            continue
    
    raise Exception(f"读取CSV文件失败: 尝试了多种编码方式({', '.join(encodings)})均失败")


def write_csv_file(file_path: str, headers: List[str], rows: List[List[str]]):
    """通用CSV文件写入函数"""
    with open(file_path, "w", encoding="utf-8", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(headers)
        for row in rows:
            writer.writerow(row)

## 数据格式化函数

In [72]:
def format_number_value(value: Any) -> str:
    """格式化数值，保留完整的小数位数，避免精度丢失"""
    if value is None:
        return ""
    
    if isinstance(value, float):
        if value == int(value):
            return str(int(value))
        
        value_str = repr(value)
        
        if 'e' in value_str.lower() or 'E' in value_str:
            formatted = f"{value:.17f}"
            formatted = formatted.rstrip('0')
            if formatted.endswith('.'):
                formatted = formatted[:-1]
            return formatted
        
        return value_str
    
    if isinstance(value, int):
        return str(value)
    
    return str(value)


def format_time_with_milliseconds(time_str: str) -> str:
    """格式化时间字符串，确保包含毫秒部分"""
    if not time_str or not time_str.strip():
        return time_str
    
    time_str = time_str.strip()
    
    if 'T' in time_str:
        time_str = time_str.replace('T', ' ')
    
    if '.' in time_str:
        parts = time_str.split('.')
        if len(parts) == 2:
            milliseconds = parts[1]
            if len(milliseconds) < 3:
                milliseconds = milliseconds.ljust(3, '0')
            elif len(milliseconds) > 3:
                milliseconds = milliseconds[:3]
            return f"{parts[0]}.{milliseconds}"
        return time_str
    
    if len(time_str) >= 19:
        return f"{time_str}.000"
    elif len(time_str) >= 10:
        return f"{time_str} 00:00:00.000"
    
    return time_str


def format_time_for_excel(value: str, header: str = "") -> str:
    """格式化时间值以便在Excel/WPS中正确显示"""
    if not value or value == "null" or value.strip() == "":
        return value
    
    value_str = str(value).strip()
    header_lower = header.lower() if header else ""
    
    duration_keywords = ["interval", "timespan", "millisecond", "second", "duration", 
                        "elapsed", "span", "diff", "delta", "between"]
    is_duration_field = any(keyword in header_lower for keyword in duration_keywords)
    
    if is_duration_field:
        return value_str
    
    is_pure_number = bool(re.match(r'^-?\d+(\.\d+)?$', value_str))
    if is_pure_number:
        return value_str
    
    is_time_field = ("time" in header_lower or "date" in header_lower) and not is_duration_field
    
    time_pattern = r'^\d{4}-\d{2}-\d{2}(\s+\d{2}:\d{2}:\d{2}(\.\d+)?)?'
    is_time_format = bool(re.match(time_pattern, value_str))
    
    if is_time_format:
        return format_time_with_milliseconds(value_str)
    
    return value_str

## 数据处理函数

In [73]:
def flatten_dict(d: Dict[str, Any]) -> Dict[str, Any]:
    """打平嵌套字典，直接使用JSON中的特征名"""
    if d is None or not isinstance(d, dict):
        return {}
    
    items = []
    for k, v in d.items():
        clean_key = k
        
        if isinstance(v, dict):
            flattened = flatten_dict(v)
            if flattened:
                items.extend(flattened.items())
            else:
                items.append((clean_key, v))
        else:
            items.append((clean_key, v))
    return dict(items)


def compare_values(value1: Any, value2: Any) -> bool:
    """比较两个值是否一致"""
    val1_str = str(value1).strip() if value1 is not None else ""
    val2_str = str(value2).strip() if value2 is not None else ""
    
    val1_str_check = val1_str.lower()
    val1_null = val1_str == "" or val1_str_check in ["null", "none"]

    val2_str_check = val2_str.lower()
    val2_null = val2_str == "" or val2_str_check in ["null", "none"]

    if val1_null and val2_null:
        return True

    if val1_null or val2_null:
        return False

    try:
        val1_num = float(val1_str)
        val2_num = float(val2_str)
        return val1_num == val2_num
    except (ValueError, TypeError):
        return val1_str == val2_str

## 设置文件路径

In [74]:
# 设置输入输出文件路径
script_dir = os.getcwd()

# 输入文件
input_json_csv = os.path.join(script_dir, "qx_json_a.csv")  # 原始在线JSON数据
offline_csv = os.path.join(script_dir, "qx_23_a.csv")  # 离线SQL数据

# 输出文件
output_compare_csv = os.path.join(script_dir, "bz_0911_order.csv")
parsed_online_csv = output_compare_csv.replace('.csv', '_线上格式化数据.csv')

# JSON列名配置
json_column_names = ["local_olduser_uncomplete_interface"]  # 需要解析的JSON列名列表


In [75]:
# 离线数据
sqldata=pd.read_csv('qx_23_a.csv') 
sqldata

,ua_id,cust_no,ua_time,local_olduser_uncompleted_mincreatedhour_v2,local_olduser_uncompleted_avgcreatedhour_v2,local_olduser_uncompleted_maxcreatedhour_v2,local_olduser_uncompleted_uncompletedordercnt_v2,local_olduser_uncompleted_uncompletedsundayordercnt_v2,local_olduser_uncompleted_uncompletedmultiordercnt_v2,local_olduser_uncompleted_uncompletedmultiorderratio_v2,...,local_olduser_uncompleted_riskvo60d_diffratiouncompletedandcompletedamount_v2,local_olduser_uncompleted_riskvo60d_ratiouncompletedandcompletedcnt_v2,local_olduser_uncompleted_riskvo180d_instalmentcnt_v2,local_olduser_uncompleted_riskvo180d_instalmentcompletedcnt_v2,local_olduser_uncompleted_riskvo180d_instalmentuncompletedcnt_v2,local_olduser_uncompleted_riskvo180d_ratiocompletedandtotalamount_v2,local_olduser_uncompleted_riskvo180d_ratiouncompletedandtotalamount_v2,local_olduser_uncompleted_riskvo180d_ratiouncompletedandtotalcnt_v2,local_olduser_uncompleted_riskvo180d_diffratiouncompletedandcompletedamount_v2,local_olduser_uncompleted_riskvo180d_ratiouncompletedandcompletedcnt_v2
0,1210876934309167104,800001651766,2026-01-12 21:12:37.749000a,NaN,NaN,NaN,0,0,0,NaN,...,NaN,NaN,0,0,0,NaN,NaN,NaN,NaN,NaN
1,1210874254249566208,800000779142,2026-01-12 21:11:19.763000a,9.0,17.277778,23.0,6,3,17,0.944444,...,1.000000,NaN,12,0,12,0.000000,1.000000,1.000000,1.000000,NaN
2,1210873395256115200,800001553944,2026-01-12 21:10:54.969000a,15.0,19.000000,23.0,2,0,1,0.500000,...,1.000000,NaN,10,0,10,0.000000,1.000000,1.000000,1.000000,NaN
3,1210872845500293120,800001660753,2026-01-12 21:10:38.011000a,NaN,NaN,NaN,0,0,0,NaN,...,NaN,NaN,0,0,0,NaN,NaN,NaN,NaN,NaN
4,1210871745988673536,800001337958,2026-01-12 21:10:06.718000a,2.0,10.500000,19.0,1,0,1,0.500000,...,1.000000,NaN,1,0,1,0.000000,1.000000,1.000000,1.000000,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2499,1209318548375486464,800001554586,2026-01-12 08:36:42.683000a,9.0,9.000000,9.0,2,1,1,0.500000,...,0.862069,7.0,14,1,13,0.039409,0.960591,0.928571,0.921182,13.0
2500,1209317655022280704,800001652413,2026-01-12 08:36:16.945000a,17.0,17.000000,17.0,1,0,0,0.000000,...,1.000000,NaN,8,0,8,0.000000,1.000000,1.000000,1.000000,NaN
2501,1209317036546998272,800001150658,2026-01-12 08:35:58.351000a,6.0,14.333333,21.0,10,2,14,0.933333,...,0.962108,27.0,28,1,27,0.018946,0.981054,0.964286,0.962108,27.0
2502,1209316967827505152,800001425991,2026-01-12 08:35:56.385000a,9.0,13.000000,18.0,7,2,6,0.857143,...,1.000000,NaN,17,0,17,0.000000,1.000000,1.000000,1.000000,NaN


In [76]:
# 解析后在线数据
onlinedata=pd.read_csv(output_compare_csv.replace('.csv', '_线上格式化数据.csv')) 
onlinedata

,apply_id,cust_no,create_time,local_olduser_uncompleted_avgcreatedhour_v2,local_olduser_uncompleted_avgcreditusageratio_effective_v2,local_olduser_uncompleted_avgcreditusageratio_fixed_v2,local_olduser_uncompleted_maxcreatedhour_v2,local_olduser_uncompleted_maxcreditusageratio_effective_v2,local_olduser_uncompleted_maxcreditusageratio_fixed_v2,local_olduser_uncompleted_mincreatedhour_v2,...,local_olduser_uncompleted_riskvo60d_instalmentcompletedcnt_v2,local_olduser_uncompleted_riskvo60d_instalmentuncompletedcnt_v2,local_olduser_uncompleted_riskvo60d_ratiocompletedandtotalamount_v2,local_olduser_uncompleted_riskvo60d_ratiouncompletedandcompletedcnt_v2,local_olduser_uncompleted_riskvo60d_ratiouncompletedandtotalamount_v2,local_olduser_uncompleted_riskvo60d_ratiouncompletedandtotalcnt_v2,local_olduser_uncompleted_uncompletedmultiordercnt_v2,local_olduser_uncompleted_uncompletedmultiorderratio_v2,local_olduser_uncompleted_uncompletedordercnt_v2,local_olduser_uncompleted_uncompletedsundayordercnt_v2
0,1210850786548269056,800001224685,2026-01-12 20:59:58a.000,17.666667,0.348869,0.348869,23.0,1.000000,1.000000,13.0,...,4,9,0.281955,2.25,0.718045,0.692308,5,0.833333,5,2
1,1210850099353485312,800001143933,2026-01-12 20:59:39a.000,12.333333,0.534915,0.534915,21.0,1.000000,1.000000,1.0,...,0,9,0.000000,NaN,1.000000,1.000000,8,0.888889,4,1
2,1210849755756101632,800001079004,2026-01-12 20:59:29a.000,13.923077,0.271509,0.271509,21.0,1.000000,1.000000,8.0,...,2,14,0.038070,7.00,0.961930,0.875000,12,0.923077,7,0
3,1210849480878194688,800001254286,2026-01-12 20:59:20a.000,14.222222,0.417892,0.417892,21.0,1.000000,1.000000,9.0,...,0,16,0.000000,NaN,1.000000,1.000000,8,0.888889,7,0
4,1210849240360042496,800000827535,2026-01-12 20:59:14a.000,18.230769,0.197429,0.197429,23.0,1.000000,1.000000,8.0,...,4,29,0.035179,7.25,0.964821,0.878788,25,0.961538,13,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2975,1209318170418348032,800000811589,2026-01-12 08:36:37a.000,13.727273,0.229208,0.229208,19.0,1.000000,1.000000,2.0,...,0,19,0.000000,NaN,1.000000,1.000000,21,0.954545,8,4
2976,1209317036546998272,800001150658,2026-01-12 08:35:59a.000,14.642857,0.258326,0.258326,21.0,1.000000,1.000000,6.0,...,0,27,0.000000,NaN,1.000000,1.000000,13,0.928571,10,2
2977,1209316967827505152,800001425991,2026-01-12 08:35:58a.000,13.333333,0.299606,0.299606,18.0,0.791667,0.791667,9.0,...,0,17,0.000000,NaN,1.000000,1.000000,5,0.833333,6,2
2978,1209316521150914560,800000859636,2026-01-12 08:35:45a.000,12.000000,0.348501,0.348501,17.0,1.000000,1.000000,7.0,...,0,13,0.000000,NaN,1.000000,1.000000,9,0.900000,5,1


## 步骤1：解析JSON数据

In [91]:
def parse_json_to_csv(input_csv_path: str, output_csv_path: str):
    """解析CSV文件，提取并打平JSON字段"""
    print(f"步骤1: 开始解析JSON数据")
    print(f"输入文件: {input_csv_path}")
    print(f"输出文件: {output_csv_path}")
    
    # 读取CSV文件
    base_headers, rows = read_csv_with_encoding(input_csv_path)
    
    print(f"读取完成，共 {len(rows)} 行数据")
    
    # 找到JSON列的索引
    json_column_index = None
    
    for i, header in enumerate(base_headers):
        if header in json_column_names:
            json_column_index = i
            json_column_name = header
            break
    
    if json_column_index is None:
        raise ValueError(f"未找到JSON列，尝试查找的列名: {json_column_names}")
    
    print(f"找到JSON字段: {json_column_name} (索引: {json_column_index})")
    
    # 解析JSON并收集所有特征字段
    all_feature_keys = set()
    parsed_data = []
    
    print(f"开始解析JSON字段...")
    for row_index, row in enumerate(rows):
        if row_index % 1000 == 0:
            print(f"已处理: {row_index}/{len(rows)}")
        
        # 获取基础信息（除了JSON列）
        base_data = {}
        for i, header in enumerate(base_headers):
            if i != json_column_index:
                value = row[i] if i < len(row) else ""
                if header in ["create_time", "use_create_time", "apply_time", "update_time"]:
                    value = format_time_with_milliseconds(value)
                base_data[header] = value
        
        # 解析JSON字段
        json_str = row[json_column_index] if json_column_index < len(row) else ""
        
        if not json_str or not json_str.strip() or json_str.strip().lower() in ["null", "none"]:
            continue
        
        try:
            json_obj = json.loads(json_str)
            
            if json_obj is None or not isinstance(json_obj, dict):
                if json_obj is not None:
                    print(f"警告: 第 {row_index + 2} 行JSON解析结果不是字典类型，跳过该客户")
                continue
            
            # 打平嵌套字典
            flattened = flatten_dict(json_obj)
            
            # 收集所有特征字段
            all_feature_keys.update(flattened.keys())
            
            # 合并基础信息和特征
            merged_data = {**base_data, **flattened}
            parsed_data.append(merged_data)
            
        except json.JSONDecodeError as e:
            print(f"警告: 第 {row_index + 2} 行JSON解析失败: {str(e)}，跳过该客户")
            continue
        except Exception as e:
            print(f"警告: 第 {row_index + 2} 行处理JSON时发生错误: {str(e)}，跳过该客户")
            continue
    
    print(f"解析完成，共发现 {len(all_feature_keys)} 个特征字段")
    
    # 构建完整的表头
    feature_keys_sorted = sorted(all_feature_keys)
    all_headers = []
    
    # 先添加基础列（排除JSON列）
    for header in base_headers:
        if header != json_column_name:
            all_headers.append(header)
    
    # 再添加特征列
    all_headers.extend(feature_keys_sorted)
    
    print(f"总列数: {len(all_headers)} (基础列: {len(base_headers) - 1}, 特征列: {len(feature_keys_sorted)})")
    
    # 准备写入数据
    output_rows = []
    for parsed_row in parsed_data:
        row_data = []
        for header in all_headers:
            value = parsed_row.get(header, "")
            if value is None:
                value = ""
            
            if isinstance(value, (int, float)):
                value_str = format_number_value(value)
            else:
                value_str = str(value)
            
            value_str = format_time_for_excel(value_str, header)
            row_data.append(value_str)
        output_rows.append(row_data)
    
    # 写入新的CSV文件
    # print(f"开始写入输出文件: {output_csv_path}")
    write_csv_file(output_csv_path, all_headers, output_rows)
    
    print(f"JSON解析完成: {output_csv_path}")
    print(f"共写入 {len(parsed_data)} 行数据")
    
    return output_csv_path

# 执行JSON解析
try:
    parsed_file = parse_json_to_csv(input_json_csv, parsed_online_csv)
    print(f"✓ JSON解析完成: {parsed_file}")
except Exception as e:
    print(f"✗ JSON解析失败: {str(e)}")
    import traceback
    traceback.print_exc()

步骤1: 开始解析JSON数据
输入文件: /home/jovyan/work/DataCheck/qx_json_a.csv
输出文件: /home/jovyan/work/DataCheck/bz_0911_order_线上格式化数据.csv
文件读取成功: /home/jovyan/work/DataCheck/qx_json_a.csv, 使用编码: utf-8, 共 2980 行
读取完成，共 2980 行数据
找到JSON字段: local_olduser_uncomplete_interface (索引: 3)
开始解析JSON字段...
已处理: 0/2980
已处理: 1000/2980
已处理: 2000/2980
解析完成，共发现 29 个特征字段
总列数: 32 (基础列: 3, 特征列: 29)
JSON解析完成: /home/jovyan/work/DataCheck/bz_0911_order_线上格式化数据.csv
共写入 2980 行数据
✓ JSON解析完成: /home/jovyan/work/DataCheck/bz_0911_order_线上格式化数据.csv


## 步骤2：数据对比分析

In [92]:
def write_all_comparison_details(output_path: str, differences_dict: Dict, matches_dict: Dict, all_features: List[str]):
    """写入所有对比数据明细文件"""
    # print(f"开始写入所有对比数据明细文件: {output_path}")
    
    # 收集所有客户数据
    customer_data = {}
    
    # 处理差异数据
    for (apply_id, feature), (online_value, offline_value, cust_no, create_time) in differences_dict.items():
        customer_key = (apply_id, cust_no, create_time)
        if customer_key not in customer_data:
            customer_data[customer_key] = {}
        customer_data[customer_key][feature] = (offline_value, online_value)
    
    # 处理一致数据
    for (apply_id, cust_no, create_time, feature), value in matches_dict.items():
        customer_key = (apply_id, cust_no, create_time)
        if customer_key not in customer_data:
            customer_data[customer_key] = {}
        customer_data[customer_key][feature] = (value, value)
    
    # 构建表头
    headers = ["apply_id", "cust_no", "create_time"]
    for feature in sorted(all_features):
        headers.append(f"{feature}_x")  # sql值
        headers.append(f"{feature}_y")  # json值
    
    # 写入CSV文件
    with open(output_path, "w", encoding="utf-8", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(headers)
        
        sorted_customers = sorted(customer_data.items(), key=lambda x: (x[0][0], x[0][1], x[0][2]))
        
        rows_written = 0
        for (apply_id, cust_no, create_time), features_data in sorted_customers:
            row = [apply_id, cust_no, format_time_for_excel(str(create_time), "create_time")]
            
            for feature in sorted(all_features):
                if feature in features_data:
                    offline_value, online_value = features_data[feature]
                    offline_str = format_number_value(offline_value) if isinstance(offline_value, (int, float)) else str(offline_value).strip() if offline_value is not None else ""
                    online_str = format_number_value(online_value) if isinstance(online_value, (int, float)) else str(online_value).strip() if online_value is not None else ""
                    row.append(offline_str)
                    row.append(online_str)
                else:
                    row.append("")
                    row.append("")
            
            writer.writerow(row)
            rows_written += 1
    
    print(f"所有对比数据明细文件写入完成: {output_path}，共 {rows_written} 行数据")


def write_feature_difference_ratio(output_path: str, differences_dict: Dict, matches_dict: Dict, all_features: List[str], matched_count: int):
    """写入差异特征有差异占比文件"""
    # print(f"开始写入差异特征有差异占比文件: {output_path}")
    
    # 统计每个特征的差异情况
    feature_stats = {}
    
    # 初始化特征统计
    for feature in all_features:
        feature_stats[feature] = {
            'total_count': 0,
            'diff_count': 0,
            'match_count': 0
        }
    
    # 统计差异数据
    for (apply_id, feature), (online_value, offline_value, cust_no, create_time) in differences_dict.items():
        if feature in feature_stats:
            feature_stats[feature]['diff_count'] += 1
            feature_stats[feature]['total_count'] += 1
    
    # 统计一致数据
    for (apply_id, cust_no, create_time, feature), value in matches_dict.items():
        if feature in feature_stats:
            feature_stats[feature]['match_count'] += 1
            feature_stats[feature]['total_count'] += 1
    
    # 构建输出数据
    headers = ["特征名称", "总对比次数", "一致次数", "差异次数", "一致率(%)", "差异率(%)"]
    rows = []
    
    for feature in sorted(all_features):
        stats = feature_stats[feature]
        total = stats['total_count']
        match = stats['match_count']
        diff = stats['diff_count']
        
        if total > 0:
            match_ratio = (match / total) * 100
            diff_ratio = (diff / total) * 100
        else:
            match_ratio = 0
            diff_ratio = 0
        
        rows.append([
            feature,
            total,
            match,
            diff,
            f"{match_ratio:.2f}",
            f"{diff_ratio:.2f}"
        ])
    
    # 写入CSV文件
    write_csv_file(output_path, headers, rows)
    print(f"差异特征有差异占比文件写入完成: {output_path}，共 {len(rows)} 个特征")


def write_difference_details(output_path: str, differences_dict: Dict, all_features: List[str]):
    """写入差异明细文件"""
    # print(f"开始写入差异明细文件: {output_path}")
    
    # 构建表头
    headers = ["apply_id", "cust_no", "create_time", "特征名称", "sql值", "json值"]
    rows = []
    
    # 按apply_id和特征名称排序
    sorted_differences = sorted(differences_dict.items(), key=lambda x: (x[0][0], x[0][1]))
    
    for (apply_id, feature), (online_value, offline_value, cust_no, create_time) in sorted_differences:
        # 格式化值
        offline_str = format_number_value(offline_value) if isinstance(offline_value, (int, float)) else str(offline_value).strip() if offline_value is not None else ""
        online_str = format_number_value(online_value) if isinstance(online_value, (int, float)) else str(online_value).strip() if online_value is not None else ""
        
        rows.append([
            apply_id,
            cust_no,
            format_time_for_excel(str(create_time), "create_time"),
            feature,
            offline_str,
            online_str
        ])
    
    # 写入CSV文件
    write_csv_file(output_path, headers, rows)
    print(f"差异明细文件写入完成: {output_path}，共 {len(rows)} 条差异记录")


def compare_csv_files(file1_path: str, file2_path: str, output_path: str):
    """对比两个CSV文件中的特征值"""
    print(f"步骤2: 开始对比两个CSV文件")
    print(f"文件1 (在线): {file1_path}")
    print(f"文件2 (离线): {file2_path}")
    
    # 读取两个文件
    headers1, rows1 = read_csv_with_encoding(file1_path)
    headers2, rows2 = read_csv_with_encoding(file2_path)
    
    # 找到主键列的索引
    apply_id_idx1 = headers1.index("apply_id") if "apply_id" in headers1 else None
    
    apply_id_idx2 = None
    apply_id_column_name2 = None
    if "apply_id" in headers2:
        apply_id_idx2 = headers2.index("apply_id")
        apply_id_column_name2 = "apply_id"
    elif "use_credit_apply_id" in headers2:
        apply_id_idx2 = headers2.index("use_credit_apply_id")
        apply_id_column_name2 = "use_credit_apply_id"
    elif "ua_id" in headers2:
        apply_id_idx2 = headers2.index("ua_id")
        apply_id_column_name2 = "ua_id"
    
    if apply_id_idx1 is None:
        raise ValueError(f"文件1中未找到 apply_id 列")
    
    if apply_id_idx2 is None:
        raise ValueError(f"文件2中未找到 apply_id、use_credit_apply_id 或 ua_id 列")
    
    print(f"文件1主键列: apply_id (索引{apply_id_idx1})")
    print(f"文件2主键列: {apply_id_column_name2} (索引{apply_id_idx2})")
    
    # 获取其他必要字段的索引
    cust_no_idx1 = headers1.index("cust_no") if "cust_no" in headers1 else None
    cust_no_idx2 = headers2.index("cust_no") if "cust_no" in headers2 else None
    time_idx1 = headers1.index("create_time") if "create_time" in headers1 else None
    
    time_idx2 = None
    if "use_create_time" in headers2:
        time_idx2 = headers2.index("use_create_time")
    elif "ua_time" in headers2:
        time_idx2 = headers2.index("ua_time")
    
    # 从D列（索引3）开始对比特征
    FEATURE_START_INDEX1 = 3
    FEATURE_START_INDEX2 = 3
    
    feature_cols1_all = headers1[FEATURE_START_INDEX1:] if len(headers1) > FEATURE_START_INDEX1 else []
    feature_cols1 = [col for col in feature_cols1_all if col != "success"]
    feature_cols2 = headers2[FEATURE_START_INDEX2:] if len(headers2) > FEATURE_START_INDEX2 else []
    
    # 构建文件2的索引字典
    file2_index = {}
    for row_idx, row in enumerate(rows2):
        if apply_id_idx2 < len(row) and row[apply_id_idx2] is not None:
            apply_id = row[apply_id_idx2].strip()
            if apply_id:
                file2_index[apply_id] = (row_idx, row)
    
    print(f"文件2索引构建完成，共 {len(file2_index)} 条记录")
    
    # 构建特征名映射
    feature_mapping = {}
    all_features = []
    
    for idx, feature2 in enumerate(feature_cols2):
        actual_idx2 = FEATURE_START_INDEX2 + idx
        actual_idx1 = None
        
        for idx1, feature1 in enumerate(feature_cols1):
            if feature2 == feature1:
                actual_idx1 = FEATURE_START_INDEX1 + idx1
                break
        
        feature_mapping[feature2] = (actual_idx2, actual_idx1)
        if feature2 not in all_features:
            all_features.append(feature2)
    
    all_features = sorted(all_features)
    
    print(f"实际对比的特征数: {len(all_features)}")
    
    # 对比数据
    print(f"开始对比数据...")
    differences_dict = {}
    matches_dict = {}
    matched_count = 0
    unmatched_count = 0
    
    for row_idx1, row1 in enumerate(rows1):
        if row_idx1 % 500 == 0:
             print(f"已处理: {row_idx1}/{len(rows1)}")
        
        if apply_id_idx1 >= len(row1):
            continue
        
        apply_id1 = row1[apply_id_idx1].strip() if row1[apply_id_idx1] is not None else ""
        
        cust_no1 = row1[cust_no_idx1].strip() if cust_no_idx1 is not None and cust_no_idx1 < len(row1) and row1[cust_no_idx1] is not None else ""
        create_time1 = row1[time_idx1].strip() if time_idx1 is not None and time_idx1 < len(row1) and row1[time_idx1] is not None else ""
        
        if not apply_id1:
            unmatched_count += 1
            continue
        
        if apply_id1 not in file2_index:
            unmatched_count += 1
            continue
        
        row_idx2, row2 = file2_index[apply_id1]
        matched_count += 1
        
        cust_no2 = row2[cust_no_idx2].strip() if cust_no_idx2 is not None and cust_no_idx2 < len(row2) and row2[cust_no_idx2] is not None else ""
        use_create_time2 = row2[time_idx2].strip() if time_idx2 is not None and time_idx2 < len(row2) and row2[time_idx2] is not None else ""
        
        # 对比所有特征
        for feature_name in all_features:
            idx2, idx1 = feature_mapping.get(feature_name, (None, None))
            
            value2 = ""
            if idx2 is not None:
                if idx2 < len(row2) and row2[idx2] is not None:
                    value2 = str(row2[idx2]).strip()
            
            value1 = ""
            if idx1 is not None:
                if idx1 < len(row1) and row1[idx1] is not None:
                    value1 = str(row1[idx1]).strip()
            
            has_diff = False
            is_match = False
            if idx1 is not None and idx2 is not None:
                if compare_values(value1, value2):
                    is_match = True
                else:
                    has_diff = True
            elif idx1 is None and idx2 is not None:
                has_diff = True
            elif idx1 is not None and idx2 is None:
                has_diff = True
            
            if is_match:
                match_key = (apply_id1, cust_no2 or cust_no1, use_create_time2 or create_time1, feature_name)
                matches_dict[match_key] = value2
            
            if has_diff:
                diff_key = (apply_id1, feature_name)
                differences_dict[diff_key] = (value2, value1, cust_no2 or cust_no1, use_create_time2 or create_time1)
    
    print(f"对比完成:")
    print(f"  匹配记录数: {matched_count}")
    print(f"  未匹配记录数: {unmatched_count}")
    # print(f"  有差异的特征值数量: {len(differences_dict)}")
    
    # 计算统计信息
    total_comparisons = matched_count * len(all_features)
    diff_count = len(differences_dict)
    match_count = total_comparisons - diff_count
    match_ratio = match_count / total_comparisons * 100 if total_comparisons > 0 else 0
    
    print(f"总体统计:")
    print(f"  总对比次数: {total_comparisons}")
    # print(f"  一致数量: {match_count}")
    print(f"  差异数量: {diff_count}")
    print(f"  一致率: {match_ratio:.2f}%")
    
    # 写入所有对比数据明细文件
    matches_path = output_path.replace('.csv', '_一致数据明细.csv')
    write_all_comparison_details(matches_path, differences_dict, matches_dict, all_features)
    
    # 写入差异特征有差异占比文件
    ratio_path = output_path.replace('.csv', '_差异特征有差异占比.csv')
    write_feature_difference_ratio(ratio_path, differences_dict, matches_dict, all_features, matched_count)
    
    # 写入差异明细文件
    details_path = output_path.replace('.csv', '_差异明细.csv')
    write_difference_details(details_path, differences_dict, all_features)
    
    # 计算特征统计信息
    features_with_diff = set()
    features_without_diff = set(all_features)
    
    for (apply_id, feature), (online_value, offline_value, cust_no, create_time) in differences_dict.items():
        features_with_diff.add(feature)
        if feature in features_without_diff:
            features_without_diff.remove(feature)
    
    feature_stats = {
        'matched_records': matched_count,
        'total_features': len(all_features),
        'features_with_diff': len(features_with_diff),
        'features_without_diff': len(features_without_diff),
        'total_comparisons': total_comparisons,
        'match_count': match_count,
        'diff_count': diff_count,
        'match_ratio': match_ratio
    }
    
    return matches_path, ratio_path, details_path, feature_stats


In [93]:
# 差异占比
result=pd.read_csv(output_compare_csv.replace('.csv', '_差异特征有差异占比.csv')) 
result

,特征名称,总对比次数,一致次数,差异次数,一致率(%),差异率(%)
0,local_olduser_uncompleted_avgcreatedhour_v2,2041,58,1983,2.84,97.16
1,local_olduser_uncompleted_maxcreatedhour_v2,2041,1442,599,70.65,29.35
2,local_olduser_uncompleted_mincreatedhour_v2,2041,1489,552,72.95,27.05
3,local_olduser_uncompleted_riskvo180d_diffratio...,2041,689,1352,33.76,66.24
4,local_olduser_uncompleted_riskvo180d_instalmen...,2041,298,1743,14.60,85.40
5,local_olduser_uncompleted_riskvo180d_instalmen...,2041,1073,968,52.57,47.43
6,local_olduser_uncompleted_riskvo180d_instalmen...,2041,341,1700,16.71,83.29
7,local_olduser_uncompleted_riskvo180d_ratiocomp...,2041,689,1352,33.76,66.24
8,local_olduser_uncompleted_riskvo180d_ratiounco...,2041,885,1156,43.36,56.64
9,local_olduser_uncompleted_riskvo180d_ratiounco...,2041,689,1352,33.76,66.24


In [94]:
# 差异占比
reslutdetail=pd.read_csv(output_compare_csv.replace('.csv', '_差异明细.csv')) 
reslutdetail

,apply_id,cust_no,create_time,特征名称,离线值,在线值
0,1209316521150914560,800000859636,2026-01-12 08:35:43.926,local_olduser_uncompleted_avgcreatedhour_v2,12.000000,12.090909
1,1209316521150914560,800000859636,2026-01-12 08:35:43.926,local_olduser_uncompleted_riskvo180d_instalmen...,16.000000,19.000000
2,1209316521150914560,800000859636,2026-01-12 08:35:43.926,local_olduser_uncompleted_riskvo180d_instalmen...,16.000000,19.000000
3,1209316521150914560,800000859636,2026-01-12 08:35:43.926,local_olduser_uncompleted_riskvo60d_instalment...,13.000000,16.000000
4,1209316521150914560,800000859636,2026-01-12 08:35:43.926,local_olduser_uncompleted_riskvo60d_instalment...,13.000000,16.000000
...,...,...,...,...,...,...
30587,1210850786548269056,800001224685,2026-01-12 20:59:56.483,local_olduser_uncompleted_riskvo60d_ratiouncom...,0.692308,0.875000
30588,1210850786548269056,800001224685,2026-01-12 20:59:56.483,local_olduser_uncompleted_uncompletedmultiorde...,5.000000,6.000000
30589,1210850786548269056,800001224685,2026-01-12 20:59:56.483,local_olduser_uncompleted_uncompletedmultiorde...,0.833333,0.857143
30590,1210850786548269056,800001224685,2026-01-12 20:59:56.483,local_olduser_uncompleted_uncompletedordercnt_v2,5.000000,3.000000
